In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_DIR = Path("../Results/vary_eps")
TAG = "n200_r3"

files = sorted(glob.glob(str(RESULTS_DIR / f"{TAG}_task*.csv")))
if len(files) == 0:
    raise FileNotFoundError(f"No task CSVs found in {RESULTS_DIR} matching {TAG}_task*.csv")

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df.to_csv(RESULTS_DIR / f"{TAG}_merged.csv", index=False)

# --- sanity check for required columns
needed = {"eps", "nmse_ridge_np", "nmse_mle_np", "nmse_loc", "nmse_cen"}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in merged df: {missing}\n"
                     f"Make sure vary_eps.py writes nmse_ridge_np and nmse_mle_np.")

# Summary table
summary = (
    df.groupby("eps")
      .agg(
          ridge_np_mean=("nmse_ridge_np", "mean"),
          ridge_np_se=("nmse_ridge_np", lambda x: x.std(ddof=1) / np.sqrt(len(x))),

          mle_np_mean=("nmse_mle_np", "mean"),
          mle_np_se=("nmse_mle_np", lambda x: x.std(ddof=1) / np.sqrt(len(x))),

          loc_mean=("nmse_loc", "mean"),
          loc_se=("nmse_loc", lambda x: x.std(ddof=1) / np.sqrt(len(x))),

          cen_mean=("nmse_cen", "mean"),
          cen_se=("nmse_cen", lambda x: x.std(ddof=1) / np.sqrt(len(x))),

          count=("nmse_cen", "size"),
      )
      .reset_index()
      .sort_values("eps")
)

summary.to_csv(RESULTS_DIR / f"{TAG}_summary.csv", index=False)

# ---- Difference (privacy cost) plots
# Local vs ridge baseline
summary["loc_excess"] = summary["loc_mean"] - summary["ridge_np_mean"]

# Central vs unregularized MLE baseline
summary["cen_excess"] = summary["cen_mean"] - summary["mle_np_mean"]

x = summary["eps"].to_numpy()

# Plot 1: Local privacy cost
plt.figure()
plt.plot(np.log(x), np.log(summary["loc_excess"]), marker="s", linestyle="-",
         label="Local DP excess error: nmse_loc - nmse_ridge_np")
plt.axhline(0.0, linestyle="--", linewidth=1.0)
plt.xlabel("Privacy level µ")
plt.ylabel("Excess nMSE")
plt.title("Local DP privacy cost (relative to non-private ridge baseline)")
plt.legend()
plt.grid(True, linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{TAG}_local_excess_vs_eps.png", dpi=200)

# Plot 2: Central privacy cost
plt.figure()
plt.plot(np.log(x), np.log(summary["cen_excess"]), marker="^", linestyle="-",
         label="Central DP excess error: nmse_cen - nmse_mle_np")
plt.axhline(0.0, linestyle="--", linewidth=1.0)
plt.xlabel("Privacy level µ")
plt.ylabel("Excess nMSE")
plt.title("Central DP privacy cost (relative to non-private MLE baseline)")
plt.legend()
plt.grid(True, linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{TAG}_central_excess_vs_eps.png", dpi=200)

plt.show()

In [ ]:
# Adjust global font sizes for publication-quality figures
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14
})

plt.figure(figsize=(8, 6), dpi=300)  # High resolution for publication

# Data extraction
x = summary["eps"].to_numpy()
local_y = summary["loc_excess"].to_numpy()
central_y = summary["cen_excess"].to_numpy()

# Plot Local DP line
plt.plot(np.log10(x), np.log10(local_y), 
         marker="s", markersize=6, linestyle="-", color="tab:blue",
         linewidth=1.5, label="Local DP (Excess Error)")

# Plot Central DP line
plt.plot(np.log10(x), np.log10(central_y), 
         marker="^", markersize=7, linestyle="-", color="tab:orange",
         linewidth=1.5, label="Central DP (Excess Error)")

# Reference line for zero excess error
plt.axhline(0.0, color="black", linestyle="--", linewidth=1.0, alpha=0.5)

plt.xlabel(r"$\log_{10}(\varepsilon)$", fontsize=13)
plt.ylabel(r"$\log_{10}(\text{Excess nMSE})$", fontsize=13)

plt.title(f"Privacy Cost Comparison: Local vs. Central\n(n=200)", fontsize=14, fontweight='bold')

plt.legend(frameon=True)
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()

plot_path = RESULTS_DIR / f"{TAG}_comparison_epsilon_highres.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
import glob
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("../Results/vary_eps")
n_values = [50, 100, 150, 200, 250]
TARGET_EPS = 0.01
R = 3  # hypergraph order

all_n_data = []

for n in n_values:
    # Match files for this specific n
    pattern = str(RESULTS_DIR / f"n{n}_r{R}_task*.csv")
    files = glob.glob(pattern)
    
    if not files:
        print(f"Warning: No files found for n={n}")
        continue
    
    # Load and combine all replicates for this n
    df_n = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    
    # Filter for the specific epsilon of interest
    df_eps = df_n[np.isclose(df_n["eps"], TARGET_EPS)].copy()
    
    if df_eps.empty:
        print(f"Warning: epsilon={TARGET_EPS} not found in data for n={n}")
        continue

    # Calculate excess error per replicate
    df_eps["loc_excess"] = df_eps["nmse_loc"] - df_eps["nmse_ridge_np"]
    df_eps["cen_excess"] = df_eps["nmse_cen"] - df_eps["nmse_mle_np"]
    
    # Aggregate statistics across replicates
    res = {
        "n": n,
        "eps": TARGET_EPS,
        "loc_excess_mean": df_eps["loc_excess"].mean(),
        "loc_excess_se": df_eps["loc_excess"].std(ddof=1) / np.sqrt(len(df_eps)),
        "cen_excess_mean": df_eps["cen_excess"].mean(),
        "cen_excess_se": df_eps["cen_excess"].std(ddof=1) / np.sqrt(len(df_eps)),
        "sample_size": len(df_eps)
    }
    all_n_data.append(res)

# Create the summary DataFrame
scaling_summary = pd.DataFrame(all_n_data)
scaling_summary.to_csv(RESULTS_DIR / "scaling_n_summary_eps0.01.csv", index=False)

print("Scaling summary created:")
print(scaling_summary)

In [ ]:
# Publication-style plot formatting
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'text.usetex': False  
})

plt.figure(figsize=(8, 6), dpi=300)

# 1. Plot the actual data
plt.errorbar(scaling_summary["n"], scaling_summary["loc_excess_mean"], 
             yerr=scaling_summary["loc_excess_se"], 
             label="Local DP Excess Error", marker='s', capsize=5, linestyle='--', color='tab:blue')

plt.errorbar(scaling_summary["n"], scaling_summary["cen_excess_mean"], 
             yerr=scaling_summary["cen_excess_se"], 
             label="Central DP Excess Error", marker='^', capsize=5, linestyle='-', color='tab:red')

# 2. Log-log scales
plt.xscale('log')
plt.yscale('log')

# 3. Label ticks with log10(value) rather than the raw value
ax = plt.gca()
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{np.log10(x):.1f}'))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, pos: f'{np.log10(y):.1f}'))

plt.xticks(scaling_summary["n"])

# Reference line for zero excess error
plt.axhline(0.0, color="black", linestyle="--", linewidth=1.0, alpha=0.5)

plt.xlabel(r"$\log_{10}(n)$")
plt.ylabel(r"$\log_{10}(\text{Excess nMSE})$")

plt.title(f"Privacy Cost Comparison: Local vs. Central\n($\epsilon=0.01$)", fontsize=14, fontweight='bold')
plt.legend(frameon=True)
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()

plot_path = RESULTS_DIR / f"{TAG}_comparison_epsilon_highres.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')

plt.show()
